# Project

## Dubai International Airport (DXB)

# Team Members
    1. Omar Ayman Hassan
    2. Mohamed Maher

In [1]:
# import libraries
# importing numpy for numerical operations

import numpy as np

# Importing statistical Tests

In [2]:
from scipy import stats
# importing matplotlib for plotting
import matplotlib.pyplot as plt
# importing seaborn for enhanced visualization
import seaborn as sns


In [ ]:
from pyomo.environ import (
    ConcreteModel,
    ConstraintList,
    Binary,
    Objective,
    SolverFactory,
    Var,
    minimize,
    value,
    TerminationCondition 
)
import pandas as pd
from datetime import datetime, timedelta

#constraint paramters
min_rest_pilot = 48 
min_rest_cabin = 24
max_hours_pilot = 100
max_hours_cabin = 150

# Initialize the model
model = ConcreteModel()
# Solver path
solver = SolverFactory("cbc")

#  Parameters 
crew_df = pd.read_csv(r'd:\ITI\DS Track\Operational Research\Project\crew.csv')
flight_pairs_df = pd.read_csv(r'd:\ITI\DS Track\Operational Research\Project\flight_pairs.csv')
flight_pairs_df = flight_pairs_df
for col in ['OutDepTime', 'OutArrTime', 'InDepTime', 'InArrTime']:
    flight_pairs_df[col] = pd.to_datetime(flight_pairs_df[col])

# Calculate flight durations
flight_pairs_df['OutboundDuration'] = (flight_pairs_df['OutArrTime'] - flight_pairs_df['OutDepTime']).dt.total_seconds() / 3600
flight_pairs_df['InboundDuration'] = (flight_pairs_df['InArrTime'] - flight_pairs_df['InDepTime']).dt.total_seconds() / 3600
flight_pairs_df['TotalDuration'] = flight_pairs_df['OutboundDuration'] + flight_pairs_df['InboundDuration']


# Crew sets
pilots = crew_df[crew_df['Role'] == 'Pilot']['CrewID'].tolist()
cabin_crew = crew_df[crew_df['Role'] == 'Cabin']['CrewID'].tolist()
all_crew = pilots + cabin_crew
flight_pairs = flight_pairs_df['PairID'].tolist()

crew_data = {c: {
    'role': crew_df.loc[crew_df['CrewID'] == c, 'Role'].values[0],
    'qualified': crew_df.loc[crew_df['CrewID'] == c, 'QualifiedAircraft'].values[0].split(','),
    'hourly_rate': crew_df.loc[crew_df['CrewID'] == c, 'HourlyRate'].values[0],
    'max_pairs': crew_df.loc[crew_df['CrewID'] == c, 'MaxPairsPerMonth'].values[0]
} for c in all_crew}

flight_data = {p: {
    'aircraft': flight_pairs_df.loc[flight_pairs_df['PairID'] == p, 'OutAircraft'].values[0],
    'duration': flight_pairs_df.loc[flight_pairs_df['PairID'] == p, 'TotalDuration'].values[0],
    'departure': flight_pairs_df.loc[flight_pairs_df['PairID'] == p, 'OutDepTime'].values[0],
    'arrival': flight_pairs_df.loc[flight_pairs_df['PairID'] == p, 'InArrTime'].values[0]
} for p in flight_pairs}



# Dictionaries
crew_qualified = crew_df.set_index('CrewID')['QualifiedAircraft'].apply(lambda x: x.split(',')).to_dict()
pair_aircraft = flight_pairs_df.set_index('PairID')['OutAircraft'].to_dict()
pair_duration = flight_pairs_df.set_index('PairID')['TotalDuration'].to_dict()
crew_hourly_rate = crew_df.set_index('CrewID')['HourlyRate'].to_dict()
crew_max_pairs = crew_df.set_index('CrewID')['MaxPairsPerMonth'].to_dict()

pair_departure = flight_pairs_df.set_index('PairID')['OutDepTime'].to_dict()
pair_arrival = flight_pairs_df.set_index('PairID')['InArrTime'].to_dict()

# Decision Variables
model.assignment = Var(all_crew, flight_pairs, within=Binary)

# Objective
def total_cost(model):
    return sum(model.assignment[c, p] * pair_duration[p] * crew_hourly_rate[c]
              for c in all_crew for p in flight_pairs)
model.objective = Objective(rule=total_cost, sense=minimize)

# Constraints
model.constraints = ConstraintList()

# 1. Coverage constraints
for p in flight_pairs:
    
    model.constraints.add(sum(model.assignment[c, p] for c in pilots) == 1)
    model.constraints.add(sum(model.assignment[c, p] for c in cabin_crew) == 2)

for c in all_crew:
    qualified_aircraft = crew_qualified[c]
    for p in flight_pairs:
        if pair_aircraft[p] not in qualified_aircraft:
            model.constraints.add(model.assignment[c, p] == 0)

# 3.pairs constraints
for c in all_crew:
    model.constraints.add(sum(model.assignment[c, p] for p in flight_pairs) <= crew_max_pairs[c])

# hours constraints
for c in all_crew:
    total_hours = sum(model.assignment[c, p] * pair_duration[p] for p in flight_pairs)
    max_hours = 100 if c in pilots else 150
    model.constraints.add(total_hours <= max_hours)

"""Simplified version matching second code's style"""

    
# 5. crew rest
pair_departure = {p: flight_data[p]['departure'] for p in flight_pairs}
pair_arrival = {p: flight_data[p]['arrival'] for p in flight_pairs}
    
for c in all_crew:
    role = crew_data[c]['role']
    if(role == 'Pilot'):
        min_rest = min_rest_pilot
    else :
        min_rest = min_rest_cabin
        
    # Sort pairs by departure time 
    pairs = sorted(flight_pairs, key=lambda x: pair_departure[x])
        
    for i in range(len(pairs)):
        f1 = pairs[i]
        for j in range(i+1, len(pairs)):
            f2 = pairs[j]
            rest_hours = pd.to_timedelta(
                flight_data[f2]['departure'] - flight_data[f1]['arrival']
            ).total_seconds()/3600
            if 0 < rest_hours < min_rest:
                model.constraints.add(model.assignment[c, f1] + model.assignment[c, f2] <= 1)
                

    
    
# Solve the Model
results = solver.solve(model)

print("Optimal Solution Found\n")
print(f"Total Cost: ${value(model.objective):.2f}\n")


Optimal Solution Found

Total Cost: $144444.00



In [ ]:
from pyomo.environ import (
    ConcreteModel as CM,
    ConstraintList as CL,
    Binary,
    Objective,
    SolverFactory as SF,
    Var,
    minimize,
    value,
    TerminationCondition as TC
)
import pandas as pd
from datetime import datetime as dt, timedelta as td

optimizer = SF("cbc")

crew_data = pd.read_csv(r'd:\ITI\DS Track\Operational Research\Project\crew.csv')
flight_combinations = pd.read_csv(r'd:\ITI\DS Track\Operational Research\Project\flight_pairs.csv')

time_columns = ['OutDepTime', 'OutArrTime', 'InDepTime', 'InArrTime']
for col in time_columns:
    flight_combinations[col] = pd.to_datetime(flight_combinations[col])

flight_combinations['OutboundTime'] = (flight_combinations['OutArrTime'] - flight_combinations['OutDepTime']).dt.total_seconds() / 3600
flight_combinations['InboundTime'] = (flight_combinations['InArrTime'] - flight_combinations['InDepTime']).dt.total_seconds() / 3600
flight_combinations['CombinedDuration'] = flight_combinations['OutboundTime'] + flight_combinations['InboundTime']

captains = crew_data[crew_data['Role'] == 'Pilot']['CrewID'].tolist()
attendants = crew_data[crew_data['Role'] == 'Cabin']['CrewID'].tolist()
all_flight_sets = flight_combinations['PairID'].tolist()

crew_certifications = crew_data.set_index('CrewID')['QualifiedAircraft'].apply(lambda x: x.split(',')).to_dict()
flight_aircraft_type = flight_combinations.set_index('PairID')['OutAircraft'].to_dict()
flight_total_time = flight_combinations.set_index('PairID')['CombinedDuration'].to_dict()
crew_pay_rate = crew_data.set_index('CrewID')['HourlyRate'].to_dict()
max_assignments = crew_data.set_index('CrewID')['MaxPairsPerMonth'].to_dict()
flight_start_time = flight_combinations.set_index('PairID')['OutDepTime'].to_dict()
flight_end_time = flight_combinations.set_index('PairID')['InArrTime'].to_dict()

# Flight Officer Optimization 
officer_schedule = CM()
required_rest_officer = 48
max_duty_hours_officer = 100

# Variables
officer_schedule.crew_assignment = Var(captains, all_flight_sets, within=Binary)

# Cost function
def calculate_officer_expenses(model):
    return sum(model.crew_assignment[o, f] * flight_total_time[f] * crew_pay_rate[o]
              for o in captains for f in all_flight_sets)
officer_schedule.total_cost = Objective(rule=calculate_officer_expenses, sense=minimize)

# Constraints
officer_schedule.rules = CL()

for f in all_flight_sets:
    officer_schedule.rules.add(sum(officer_schedule.crew_assignment[o, f] for o in captains) == 1)

for o in captains:
    approved_aircraft = crew_certifications[o]
    for f in all_flight_sets:
        if flight_aircraft_type[f] not in approved_aircraft:
            officer_schedule.rules.add(officer_schedule.crew_assignment[o, f] == 0)

# Maximum assignments
for o in captains:
    officer_schedule.rules.add(sum(officer_schedule.crew_assignment[o, f] for f in all_flight_sets) <= max_assignments[o])

# Duty hour limits
for o in captains:
    total_duty = sum(officer_schedule.crew_assignment[o, f] * flight_total_time[f] for f in all_flight_sets)
    officer_schedule.rules.add(total_duty <= max_duty_hours_officer)

# Rest period enforcement
for o in captains:
    sorted_flights = sorted(all_flight_sets, key=lambda x: flight_start_time[x])
    for i in range(len(sorted_flights)):
        flight1 = sorted_flights[i]
        for j in range(i+1, len(sorted_flights)):
            flight2 = sorted_flights[j]
            rest_period = (flight_start_time[flight2] - flight_end_time[flight1]).total_seconds()/3600
            if 0 < rest_period < required_rest_officer:
                officer_schedule.rules.add(officer_schedule.crew_assignment[o, flight1] + officer_schedule.crew_assignment[o, flight2] <= 1)

# Cabin Team Optimization
cabin_team_schedule = CM()
minimum_rest_attendant = 24
maximum_duty_hours = 150

# Variables
cabin_team_schedule.staff_allocation = Var(attendants, all_flight_sets, within=Binary)

# Cost function
def calculate_cabin_costs(model):
    return sum(model.staff_allocation[a, f] * flight_total_time[f] * crew_pay_rate[a]
              for a in attendants for f in all_flight_sets)
cabin_team_schedule.operational_cost = Objective(rule=calculate_cabin_costs, sense=minimize)

# Constraints
cabin_team_schedule.requirements = CL()

# Two attendants per flight
for f in all_flight_sets:
    cabin_team_schedule.requirements.add(sum(cabin_team_schedule.staff_allocation[a, f] for a in attendants) == 2)

# Aircraft certification
for a in attendants:
    allowed_aircraft = crew_certifications[a]
    for f in all_flight_sets:
        if flight_aircraft_type[f] not in allowed_aircraft:
            cabin_team_schedule.requirements.add(cabin_team_schedule.staff_allocation[a, f] == 0)

# Assignment limits
for a in attendants:
    cabin_team_schedule.requirements.add(sum(cabin_team_schedule.staff_allocation[a, f] for f in all_flight_sets) <= max_assignments[a])

# Hour restrictions
for a in attendants:
    duty_time = sum(cabin_team_schedule.staff_allocation[a, f] * flight_total_time[f] for f in all_flight_sets)
    cabin_team_schedule.requirements.add(duty_time <= maximum_duty_hours)

# Break requirements
for a in attendants:
    ordered_flights = sorted(all_flight_sets, key=lambda x: flight_start_time[x])
    for i in range(len(ordered_flights)):
        leg1 = ordered_flights[i]
        for j in range(i+1, len(ordered_flights)):
            leg2 = ordered_flights[j]
            break_duration = (flight_start_time[leg2] - flight_end_time[leg1]).total_seconds()/3600
            if 0 < break_duration < minimum_rest_attendant:
                cabin_team_schedule.requirements.add(cabin_team_schedule.staff_allocation[a, leg1] + cabin_team_schedule.staff_allocation[a, leg2] <= 1)

# Optimization
officer_solution = optimizer.solve(officer_schedule)
cabin_solution = optimizer.solve(cabin_team_schedule)

print("Optimization Results\n")
print(f"Flight Officers Total Cost: ${value(officer_schedule.total_cost):.2f}")
print(f"Cabin Personnel Total Cost: ${value(cabin_team_schedule.operational_cost):.2f}")
print(f"\nAggregate Operational Cost: ${value(officer_schedule.total_cost) + value(cabin_team_schedule.operational_cost):.2f}")

Optimization Results

Flight Officers Total Cost: $72764.00
Cabin Personnel Total Cost: $71680.00

Aggregate Operational Cost: $144444.00


In [ ]:
from pyomo.environ import *
import pandas as pd
from datetime import datetime, timedelta

solver = SolverFactory("cbc")

# Load data
crew_data = pd.read_csv(r'd:\ITI\DS Track\Operational Research\Project\crew.csv')
flight_data = pd.read_csv(r'd:\ITI\DS Track\Operational Research\Project\flight_pairs.csv')

time_cols = ['OutDepTime', 'OutArrTime', 'InDepTime', 'InArrTime']
for col in time_cols:
    flight_data[col] = pd.to_datetime(flight_data[col])

flight_data['OutboundDuration'] = (flight_data['OutArrTime'] - flight_data['OutDepTime']).dt.total_seconds() / 3600
flight_data['InboundDuration'] = (flight_data['InArrTime'] - flight_data['InDepTime']).dt.total_seconds() / 3600
flight_data['TotalDuration'] = flight_data['OutboundDuration'] + flight_data['InboundDuration']

pilots = crew_data[crew_data['Role'] == 'Pilot']['CrewID'].tolist()
flight_pairs = flight_data['PairID'].tolist()

crew_qualifications = crew_data.set_index('CrewID')['QualifiedAircraft'].apply(lambda x: x.split(',')).to_dict()
pair_aircraft = flight_data.set_index('PairID')['OutAircraft'].to_dict()
pair_duration = flight_data.set_index('PairID')['TotalDuration'].to_dict()
crew_hourly_rate = crew_data.set_index('CrewID')['HourlyRate'].to_dict()
crew_max_assignments = crew_data.set_index('CrewID')['MaxPairsPerMonth'].to_dict()
pair_departure = flight_data.set_index('PairID')['OutDepTime'].to_dict()
pair_arrival = flight_data.set_index('PairID')['InArrTime'].to_dict()
pair_revenue = flight_data.set_index('PairID')['EstimatedRevenue'].to_dict()

# Step 1: Get original budget with relaxed regulations
original_model = ConcreteModel()
original_model.assignment = Var(pilots, flight_pairs, within=Binary)

# Objective: Minimize cost
def original_cost_rule(model):
    return sum(model.assignment[c, p] * pair_duration[p] * crew_hourly_rate[c]
              for c in pilots for p in flight_pairs)
original_model.cost = Objective(rule=original_cost_rule, sense=minimize)

# Constraints
original_model.constraints = ConstraintList()

#One pilot per flight
for p in flight_pairs:
    original_model.constraints.add(sum(original_model.assignment[c, p] for c in pilots) == 1)

# Qualification constraints
for c in pilots:
    qualified = crew_qualifications[c]
    for p in flight_pairs:
        if pair_aircraft[p] not in qualified:
            original_model.constraints.add(original_model.assignment[c, p] == 0)

# Max assignments
for c in pilots:
    original_model.constraints.add(sum(original_model.assignment[c, p] for p in flight_pairs) <= crew_max_assignments[c])

# Max hours (original: 100)
original_model.constraints.add(sum(original_model.assignment[c, p] * pair_duration[p] 
                              for c in pilots for p in flight_pairs) <= 100 * len(pilots))

# Rest constraints , original: 24 hours
for c in pilots:
    sorted_pairs = sorted(flight_pairs, key=lambda x: pair_departure[x])
    for i in range(len(sorted_pairs)):
        f1 = sorted_pairs[i]
        for j in range(i+1, len(sorted_pairs)):
            f2 = sorted_pairs[j]
            rest = (pair_departure[f2] - pair_arrival[f1]).total_seconds()/3600
            if 0 < rest < 24:
                original_model.constraints.add(original_model.assignment[c, f1] + original_model.assignment[c, f2] <= 1)

# Solve
original_result = solver.solve(original_model)
original_budget = value(original_model.cost)

# Step 2: Flight selection under new regulations
selection_model = ConcreteModel()

# Decision variables
selection_model.operate = Var(flight_pairs, within=Binary)  # 1 if flight is operated, 0 otherwise
selection_model.assignment = Var(pilots, flight_pairs, within=Binary)

# Objective: Maximize revenue
def revenue_rule(model):
    return sum(model.operate[p] * pair_revenue[p] for p in flight_pairs)
selection_model.revenue = Objective(rule=revenue_rule, sense=maximize)

# Constraints
selection_model.constraints = ConstraintList()

# Budget constraint (must match original budget)
selection_model.constraints.add(
    sum(selection_model.assignment[c, p] * pair_duration[p] * crew_hourly_rate[c]
        for c in pilots for p in flight_pairs) <= original_budget
)

# If flight is operated, assign one pilot
for p in flight_pairs:
    selection_model.constraints.add(
        sum(selection_model.assignment[c, p] for c in pilots) == selection_model.operate[p]
    )

# Qualification constraints
for c in pilots:
    qualified = crew_qualifications[c]
    for p in flight_pairs:
        if pair_aircraft[p] not in qualified:
            selection_model.constraints.add(selection_model.assignment[c, p] == 0)

# Max assignments per pilot
for c in pilots:
    selection_model.constraints.add(
        sum(selection_model.assignment[c, p] for p in flight_pairs) <= crew_max_assignments[c]
    )

# Max hours per pilot (new: 50)
for c in pilots:
    selection_model.constraints.add(
        sum(selection_model.assignment[c, p] * pair_duration[p] for p in flight_pairs) <= 50
    )

# Rest constraints (new: 48 hours)
for c in pilots:
    sorted_pairs = sorted(flight_pairs, key=lambda x: pair_departure[x])
    for i in range(len(sorted_pairs)):
        f1 = sorted_pairs[i]
        for j in range(i+1, len(sorted_pairs)):
            f2 = sorted_pairs[j]
            rest = (pair_departure[f2] - pair_arrival[f1]).total_seconds()/3600
            if 0 < rest < 48:
                selection_model.constraints.add(
                    selection_model.assignment[c, f1] + selection_model.assignment[c, f2] <= 1
                )

# Solve
selection_result = solver.solve(selection_model)

max_revenue = sum(pair_revenue[p] for p in flight_pairs)
achieved_revenue = value(selection_model.revenue)
revenue_percentage = (achieved_revenue / max_revenue) * 100


print("\nResults:")
print(f"Original pilot budget: ${original_budget:.2f}")
print(f"Maximum possible revenue: ${max_revenue:.2f}")
print(f"Achieved revenue under new regulations: ${achieved_revenue:.2f}")
print(f"Percentage of maximum revenue achieved: {revenue_percentage:.2f}%")

# Get list of flights to operate/cancel
operated_flights = [p for p in flight_pairs if value(selection_model.operate[p]) > 0.9]
canceled_flights = [p for p in flight_pairs if value(selection_model.operate[p]) < 0.1]

print("Flights to cancel:", sorted(canceled_flights))


Results:
Original pilot budget: $72764.00
Maximum possible revenue: $3876806.00
Achieved revenue under new regulations: $3854832.00
Percentage of maximum revenue achieved: 99.43%
Flights to cancel: ['P081', 'P092']
